In [ ]:
# load dataset filenames from data\train_filenames.lst and data\test_filenames.lst
train_filenames_path = "data/train_filenames.lst"
with open(train_filenames_path, "r") as f:
    train_filenames = f.read().splitlines()
test_filenames_path = "data/test_filenames.lst"
with open(test_filenames_path, "r") as f:
    test_filenames = f.read().splitlines()

print(f"Number of train images: {len(train_filenames)}")
print(f"Number of test images: {len(test_filenames)}")
print(f"Total number of images: {len(train_filenames) + len(test_filenames)}")


In [ ]:
import rasterio
import numpy as np

# 200 m tif images from csv file
images_200m = []

dataset_directory_200m = "C:\\Users\\uceda\\Documents\\uni\\master\\ENLIGHT\\course\\data\\dataset\\s2\\200m"

max_h, max_w = 21, 21

# Pad to max size
images_200m = []
for filename in train_filenames:
    with rasterio.open(dataset_directory_200m + "\\" + filename) as src:
        image = src.read()  # (12, H, W)
    pad_h = max_h - image.shape[1]
    pad_w = max_w - image.shape[2]
    # Pad with zeros on the right/bottom to make all images the same size (12, max_h, max_w)
    padded = np.pad(image, ((0, 0), (0, pad_h), (0, pad_w)), mode='constant')
    images_200m.append(padded)

images_200m = np.array(images_200m)  # shape: (N, 12, max_H, max_W)

# save to csv file
np.savetxt("images_200m.csv", images_200m.reshape(images_200m.shape[0], -1), delimiter=",")

In [ ]:
import rasterio
import random
import matplotlib.pyplot as plt
import numpy as np

def scale(band, pmin=2, pmax=98):   # to remove outliers
    lo, hi = np.percentile(band, (pmin, pmax))
    return np.clip((band - lo) / (hi - lo), 0, 1)

img = random.choice(images_200m)

band_names = ['Band 1 - Coastal aerosol', 'Band 2 - Blue', 'Band 3 - Green', 'Band 4 - Red', 'Band 5 - Vegetation red edge', 'Band 6 - Vegetation red edge', 'Band 7 - Vegetation red edge', 'Band 8 - NIR', 'Band 8A - Narrow NIR', 'Band 9 - Water vapour', 'Band 10 - SWIR - Cirrus', 'Band 11 - SWIR', 'Band 12 - SWIR']
# plot each of the 12 bands
with rasterio.open(img, 'r') as ds:
  fig, axs = plt.subplots(3, 4, figsize=(20, 15))
  print('img shape:', ds.shape)
  print('img dtype:', ds.dtypes)
  print('img count (number of bands):', ds.count)
  print('img descriptions:', ds.descriptions)
  for i in range(12):
    band = ds.read(i+1).astype(np.float32)
    axs[i//4, i%4].imshow(scale(band), cmap='viridis')
    axs[i//4, i%4].set_title(f"Band {i+1}: {band_names[i]}")
    axs[i//4, i%4].axis("off")
  plt.tight_layout()
  plt.show()

In [ ]:
species_list = [
    'Abies_alba', 
    'Acer_pseudoplatanus',
    'Alnus_spec', 
    'Betula_spec', 
    'Cleared', 
    'Fagus_sylvatica', 
    'Fraxinus_excelsior', 
    'Larix_decidua', 
    'Larix_kaempferi', 
    'Picea_abies', 
    'Pinus_nigra', 
    'Pinus_strobus', 
    'Pinus_sylvestris', 
    'Populus_spec', 
    'Prunus_spec', 
    'Pseudotsuga_menziesii', 
    'Quercus_petraea', 
    'Quercus_robur', 
    'Quercus_rubra', 
    'Tilia_spec'
]

In [ ]:
import pandas as pd
loaded_arr = np.loadtxt("images_200m.csv", delimiter=",")
b_names = [f"band_{i}" for i in range(1, 13)]
df = pd.DataFrame(loaded_arr, columns=[f"{b}_{i}" for b in b_names for i in range(1, max_h*max_w+1)])

one_hot_rows = []

for idx, filename in enumerate(train_filenames):
    species = None
    species_idx = None
    for j, s in enumerate(species_list):
        if s in filename:
            species = s
            species_idx = j
            break

    df.at[idx, 'species'] = species

    # one-hot vector: all zeros if no match, else 1 at species_idx
    one_hot_label = [1 if k == species_idx else 0 for k in range(len(species_list))]
    one_hot_rows.append(one_hot_label)

# create separate df with one vector column per sample (no individual species columns)
one_hot_df = pd.DataFrame({
    'one_hot': [row for row in one_hot_rows]
})

one_hot_df.to_csv('labels_one_hot.csv', index=False)
df.to_csv('labels_df.csv', index=False)

print(f"df shape: {df.shape}")
print(f"one_hot_df shape: {one_hot_df.shape}")
print(f"Vector length: {len(one_hot_rows[0])} (covers all {len(species_list)} species)")
one_hot_df.head()


In [ ]:
import pandas as pd
import ast
with open("labels_one_hot.csv", "r") as f:
    loaded_labels = np.array([ast.literal_eval(line.strip().strip('"')) for line in f])

# converting the array to a pandas dataframe
b_names = [f"band_{i}" for i in range(1, 13)]
max_H = 21
max_W = 21
# take into account it was reshape(3, -1) so each row is an image with 12*max_H*max_W values, we want to split it into 12 bands
df = pd.DataFrame(loaded_arr, columns=[f"{b}_{i}" for b in b_names for i in range(1, max_H*max_W+1)])
one_hot_df = pd.DataFrame(loaded_labels, columns=species_list)
df.head()
one_hot_df.head()

print(loaded_labels.shape)  # should be (N, 20)
print(loaded_labels.dtype)  # int64


In [ ]:
# CODE FOR PLOT GENERATED WITH CLAUDE -------->

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

one_hot_array = np.array(one_hot_rows)
species_counts_array = one_hot_array.sum(axis=0)

# Sort by count descending for better readability
sorted_indices = np.argsort(species_counts_array)[::-1]
sorted_counts = species_counts_array[sorted_indices]
sorted_labels = [species_list[i] for i in sorted_indices]

# Botanical color palette — earthy greens, warm neutrals, soft teals
colors = [
    "#4A7C59", "#8DB87A", "#C9DDB5",
    "#6B9E87", "#A3C4A8", "#D4E8D1",
    "#7AABB8", "#3D6B72", "#B5D5DA",
    "#9EBF8E", "#5C8A6E", "#E2EED9",
]
colors = (colors * ((len(sorted_labels) // len(colors)) + 1))[:len(sorted_labels)]

fig, (ax_pie, ax_legend) = plt.subplots(
    1, 2,
    figsize=(14, 8)
)
fig.patch.set_facecolor("#F7F5F0")

# ── Pie chart ──────────────────────────────────────────────────────────────
wedges, texts, autotexts = ax_pie.pie(
    sorted_counts,
    labels=None,               # labels go in legend panel
    autopct=lambda p: f"{p:.1f}%" if p >= 3 else "",
    pctdistance=0.78,
    colors=colors,
    startangle=140,
    wedgeprops=dict(linewidth=1.2, edgecolor="#F7F5F0"),
    shadow=False,
)

for at in autotexts:
    at.set_fontsize(10)
    at.set_color("#2C2C2C")
    at.set_fontweight("semibold")

ax_pie.set_facecolor("#F7F5F0")
ax_pie.set_aspect('equal')
ax_pie.set_title(
    "Species Distribution",
    fontsize=17, fontweight="bold",
    color="#2C2C2C", pad=20,
    fontfamily="serif"
)

# ── Legend panel ───────────────────────────────────────────────────────────
ax_legend.set_facecolor("#F7F5F0")
ax_legend.axis("off")

total = sorted_counts.sum()
y = 0.98
row_h = min(0.072, 0.96 / len(sorted_labels))   # compress if many species

for i, (label, count, color) in enumerate(zip(sorted_labels, sorted_counts, colors)):
    pct = count / total * 100

    # Color swatch
    swatch = FancyBboxPatch(
        (0.0, y - 0.030), 0.055, 0.042,
        boxstyle="round,pad=0.003",
        facecolor=color, edgecolor="#F7F5F0", linewidth=0.8,
        transform=ax_legend.transAxes, clip_on=False
    )
    ax_legend.add_patch(swatch)

    # Species name (italic)
    ax_legend.text(
        0.085, y - 0.008,
        label.replace("_", " ").capitalize(),
        transform=ax_legend.transAxes,
        fontsize=15, color="#2C2C2C",
        va="center", fontstyle="italic"
    )

    # Count + percentage (right-aligned)
    ax_legend.text(
        0.98, y - 0.008,
        f"{int(count):,}  ({pct:.1f}%)",
        transform=ax_legend.transAxes,
        fontsize=15, color="#555555",
        va="center", ha="right"
    )

    y -= row_h

# Subtle divider line between pie and legend
fig.add_artist(
    plt.Line2D([0.615, 0.615], [0.08, 0.92],
               transform=fig.transFigure,
               color="#CCCCCC", linewidth=0.8)
)

# Sample count footnote
fig.text(
    0.5, 0.02,
    f"n = {int(total):,} samples  ·  {len(sorted_labels)} species",
    ha="center", fontsize=15, color="#888888", fontstyle="italic"
)

plt.subplots_adjust(left=0.02, right=0.97, top=0.93, bottom=0.07, wspace=0.05)
plt.savefig("species_distribution.png", dpi=180, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()

In [ ]:
# Source - https://stackoverflow.com/a/68614540
# Posted by NL23codes, modified by community. See post 'Timeline' for change history
# Retrieved 2026-03-18, License - CC BY-SA 4.0

import pandas
from sklearn.decomposition import PCA
import numpy
import matplotlib.pyplot as plot
from sklearn.preprocessing import StandardScaler

# You must normalize the data before applying the fit method
#df_no_label = df.drop(columns=['species'])
df_no_label = df
scaler = StandardScaler()
X = df_no_label.values.astype(numpy.float16)
X_normalized = scaler.fit_transform(X)  # keep as numpy array, skip the DataFrame wrapper

pca = PCA(n_components=50)
pca.fit(X_normalized)

# Reformat and view results
loadings = pandas.DataFrame(
    pca.components_.T,
    columns=['PC%s' % _ for _ in range(pca.n_components_)],  # use pca.n_components_ to be safe
    index=df_no_label.columns
)
print(loadings)

plot.plot(pca.explained_variance_ratio_)
plot.ylabel('Explained Variance')
plot.xlabel('Components')
plot.show()

In [ ]:
# CODE FOR PCA GENERATED WITH CLAUDE --->
# --- Step 1: Check if 95% variance is already captured ---
cumulative_variance = numpy.cumsum(pca.explained_variance_ratio_)
n_components_90 = numpy.argmax(cumulative_variance >= 0.90) + 1

print(f"Components needed to reach 95% variance: {n_components_90}")
print(f"Variance explained by your current {pca.n_components_} components: {cumulative_variance[-1]:.4f}")

# --- Step 2: Plot cumulative variance with the 90% threshold marked ---
plot.figure(figsize=(10, 5))

plot.subplot(1, 2, 1)
plot.plot(pca.explained_variance_ratio_, marker='o', markersize=3)
plot.ylabel('Explained Variance (per component)')
plot.xlabel('Component index')
plot.title('Per-component variance')

plot.subplot(1, 2, 2)
plot.plot(cumulative_variance, marker='o', markersize=3)
plot.axhline(y=0.90, color='red', linestyle='--', label='90% threshold')
plot.axvline(x=n_components_90 - 1, color='orange', linestyle='--', label=f'{n_components_90} components')
plot.ylabel('Cumulative Explained Variance')
plot.xlabel('Number of components')
plot.title('Cumulative variance')
plot.legend()

plot.tight_layout()
plot.show()

# --- Step 3: Refit with only the components needed for 95% variance ---
pca_reduced = PCA(n_components=n_components_90)
X_reduced = pca_reduced.fit_transform(X_normalized)

print(f"\nOriginal feature count: {X.shape[1]}")
print(f"Reduced component count (95% variance): {X_reduced.shape[1]}")
print(f"Actual variance captured: {sum(pca_reduced.explained_variance_ratio_):.4f}")

In [ ]:
# --- Reduce to 45 components using the already-fitted PCA ---
X_45 = X_normalized @ pca.components_[:45].T
# (equivalent to pca.transform(X_normalized)[:, :45])

print(f"Reduced shape: {X_45.shape}")  # (n_samples, 45)

In [ ]:
import matplotlib.pyplot as plot
import numpy

# Convert one-hot back to integer class labels
species_labels = numpy.argmax(one_hot_df.values, axis=1)  # shape (45337,)

# Optional: get species names if one_hot_df has string column names
species_names = list(one_hot_df.columns)  # e.g. ['cat', 'dog', ...]

# Plot PC1 vs PC2 coloured by species
fig, ax = plot.subplots(figsize=(9, 7))

for i, name in enumerate(species_names):
    mask = species_labels == i
    ax.scatter(
        X_45[mask, 0],
        X_45[mask, 1],
        label=f'{name}',
        alpha=0.4,
        s=10
    )

var = pca.explained_variance_ratio_
ax.set_xlabel(f'PC1 ({var[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({var[1]*100:.1f}%)')
ax.set_title(
    f'PC1 vs PC2  —  {(var[0]+var[1])*100:.1f}% of total variance'
)
ax.legend(markerscale=2, bbox_to_anchor=(1.05, 1), loc='upper left')
plot.tight_layout()
plot.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# --- Compute per-row band means ---
# For each band, average across all pixels (441 values) for that row
band_means = pd.DataFrame({
    b: df[[c for c in df.columns if c.startswith(f"{b}_")]].mean(axis=1)
    for b in b_names
})
# band_means shape: (n_samples, 12)

# --- Get species labels from one-hot ---
species_labels = one_hot_df.idxmax(axis=1)  # Series of species name per row

# --- Compute mean and std per species per band ---
band_means["species"] = species_labels

stats = band_means.groupby("species")[b_names].agg(["mean", "std"])
# stats is MultiIndex columns: (band_name, "mean"/"std")

# --- Plot ---
species_list_present = stats.index.tolist()
n_species = len(species_list_present)
band_indices = np.arange(1, 13)

colors = cm.tab20(np.linspace(0, 1, n_species))

fig, ax = plt.subplots(figsize=(14, 6))

for i, species in enumerate(species_list_present):
    means = stats.loc[species, (b_names, "mean")].values
    stds  = stats.loc[species, (b_names, "std")].values

    ax.plot(band_indices, means, label=species, color=colors[i], linewidth=1.8)
    ax.fill_between(band_indices,
                    means - stds,
                    means + stds,
                    color=colors[i], alpha=0.15)

ax.set_xlabel("Band", fontsize=12)
ax.set_ylabel("Mean Reflectance (± std)", fontsize=12)
ax.set_title("Average Reflectance per Band per Species", fontsize=14)
ax.set_xticks(band_indices)
ax.set_xticklabels([f"Band {i}" for i in band_indices], rotation=45, ha="right")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8, framealpha=0.7)
ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("spectral_signatures.png", dpi=150, bbox_inches="tight")
plt.show()